# Assignment 3: Hidden Markov Models

---

## Task 1) Isolated Word Recognition

In this assignment, we'll be revising word recognition, this time using Hidden Markov Models (HMM).
As with [assignment 1](https://github.com/seqlrn/assignments/tree/master/1-dynamic-programming), we'll be using the [free spoken digits](https://github.com/Jakobovski/free-spoken-digit-dataset) dataset.
We will be using the [`pandas`](https://pandas.pydata.org/docs/) library for data handling and [`hmmlearn`](https://hmmlearn.readthedocs.io/en/latest/index.html) library for HMMs which depends on `numpy`.
Install the `pandas` and `hmmlearn` packages in your working environment and get familiar with these modules.


### Data

Download Zohar Jackson's [free spoken digit](https://github.com/Jakobovski/free-spoken-digit-dataset) dataset.
There's no need to clone, feel free to use a revision, like [v1.0.10](https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/tags/v1.0.10.tar.gz).
The file naming convention is `{digitLabel}_{speakerName}_{index}.wav`.

### Basic Setup

As you can learn from the [tutorial](https://hmmlearn.readthedocs.io/en/latest/tutorial.html#), `hmmlearn` provides us with the base implementation of Hidden Markov Models; we'll be using the `hmm.GaussianHMM`, which implements HMMs with a single Gaussian emission probability per state.
For a starter, build a basic isolated word recognizer that uses a separate model for each digit.

*In this Jupyter Notebook, we will provide the steps to solve this task and give hints via functions & comments. However, code modifications (e.g., function naming, arguments) and implementation of additional helper functions & classes are allowed. The code aims to help you get started.*

---

In [8]:
%pip install jiwer


[notice] A new release of pip is available: 25.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [9]:
# Dependencies
import os
import librosa
import numpy as np
import pandas as pd
from hmmlearn import hmm

from jiwer import wer
from itertools import groupby
from sklearn.preprocessing import StandardScaler

### Prepare the Data

1.1 To facilitate the selection of samples for speakers and digits, consider how you can store the data within a `pandas.DataFrame`.

1.2 Compute the MFCC features for the complete data set (3000 recordings; use `n_mfcc=13`).

1.3 Apply per-speaker feature normalization (e.g., standardization).

In [10]:
NUM_SAMPLES = 50 # recordings per speaker & digit
DIGITS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
SPEAKERS = ["george", "jackson", "lucas", "nicolas", "theo", "yweweler"]

In [11]:
### Notice: a good default value is 25ms for FFT window and 10ms for hop length
### Notice: be careful as librosa takes the number of samples as input!        

def compute_features(file):
    """Computes the features for a recording file."""
    y, sr = librosa.load(file, sr=None)
    
    n_fft = int(0.025 * sr)
    hop_length = int(0.01 * sr)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, hop_length=hop_length, n_fft=n_fft, n_mfcc=13)

    # return transpoed mfcc with time on x-axis
    return mfccs.transpose()


def load_dataframe(input_dir):
    """Loads the recordings into a pandas.DataFrame."""
    samples = []
    for speaker in SPEAKERS:
        for digit in DIGITS:
            for index in range(NUM_SAMPLES):
                file = "{}_{}_{}.wav".format(digit, speaker, index)
                file = os.path.join(input_dir, file)

                # compute the MFCCs
                features = compute_features(file)

                sample = {
                    "speaker": speaker, 
                    "digit": digit, 
                    "index": index,
                    "seq_length": len(features),
                    "features": features
                }
                samples.append(sample)
    
    dataframe = pd.DataFrame(data=samples)
    return dataframe


def normalize_features(dataframe):
    """Applies per-speaker feature normalization."""
    dataframe = dataframe.copy()

    scalers = {}

    for speaker in SPEAKERS:
        scaler_speaker = StandardScaler()
        data_speaker = dataframe[dataframe["speaker"]==speaker]
        features_speaker = np.concatenate(data_speaker["features"].values)
        scalers[speaker] = scaler_speaker.fit(features_speaker)

    dataframe["features"] = dataframe.apply(
        lambda row: scalers[row["speaker"]].transform(row["features"]), axis=1
    )
    return dataframe
        

In [12]:
INPUT_DIR = "<path to your recordings>"
INPUT_DIR = "data/recordings"

dataframe = load_dataframe(input_dir=INPUT_DIR)
dataframe_w_norm = normalize_features(dataframe=dataframe)

### Uncomment to use nomalized features
dataframe = dataframe_w_norm.copy()

### Notice: just for test purposes
print("Num recordings: {}".format(len(dataframe)))
for speaker in SPEAKERS:
    print("### {}".format(speaker))
    data_speaker = dataframe[dataframe["speaker"] == speaker]

    print(data_speaker["digit"].value_counts())
    print()

/home/yannes/Documents/seqlrn/.venv/lib64/python3.10/site-packages/librosa/feature/spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


Num recordings: 3000
### george
digit
0    50
1    50
2    50
3    50
4    50
5    50
6    50
7    50
8    50
9    50
Name: count, dtype: int64

### jackson
digit
0    50
1    50
2    50
3    50
4    50
5    50
6    50
7    50
8    50
9    50
Name: count, dtype: int64

### lucas
digit
0    50
1    50
2    50
3    50
4    50
5    50
6    50
7    50
8    50
9    50
Name: count, dtype: int64

### nicolas
digit
0    50
1    50
2    50
3    50
4    50
5    50
6    50
7    50
8    50
9    50
Name: count, dtype: int64

### theo
digit
0    50
1    50
2    50
3    50
4    50
5    50
6    50
7    50
8    50
9    50
Name: count, dtype: int64

### yweweler
digit
0    50
1    50
2    50
3    50
4    50
5    50
6    50
7    50
8    50
9    50
Name: count, dtype: int64



### Train and Evaluate

2.1 Implement a 6-fold [cross-validation](https://en.wikipedia.org/wiki/Cross-validation_(statistics)) loop for the 6 speakers to (later) figure out, which test speaker performs best/worst. That is, each speaker acts as test speaker while the others are used for training (with each possible combination).

2.2 Inside the cross-validation loop, train an individual HMM with linear topology for each digit. There are several points to consider:

*The [`fit`](https://github.com/hmmlearn/hmmlearn/blob/38b3cece4a6297e978a204099ae6a0a99555ec01/lib/hmmlearn/base.py#L439) expects features to be sequential in a single array with `X` as (n_train_samples, n_features). Furthermore, we need to pass the lengths of each recording into the function with`lengths` as (n_samples,):*

```python
### you can flatten the features of the train data as follows

# input: [(rec_samples_1, n_feats), ..., (rec_samples_N, n_feats)]
# output: (all_rec_samples, n_feats)
features = [features for features in dataframe["features"].values]
flatten = np.concatenate(features, axis=0)

lengths = np.array([...])

# train HMM
hmm.fit(X=flatten, lengths=lengths)
```

*For the HMM, it is necessary to choose a meaningful number of states. How many states (`n_components`) do you choose, and why?*

*With respect to the used `hmmlearn` library. How can you enforce a linear topology?*

*You might find that certain digits perform particularly bad; what could be a reason and how to mitigate it?*
    
2.3 Compute the [confusion matrix](https://en.wikipedia.org/wiki/Confusion_matrix) for each speaker and for the overall dataset by combining the predictions of the cross-validation. You can use the [`scikit-learn`](https://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html) library.

2.4 Additional experiment: Compare the results without and with per-speaker feature normalization. How does the performance change?

In [13]:
### TODO:
### 1. set the `n_components` for all digits (choose a meaningful number of states)

n_comps = {i: None for i in DIGITS}

### get average lengths for class --> HMMs will be between 5 and 7 states
n_comps = [dataframe[dataframe["digit"] == i].seq_length.mean(axis=0) for i in DIGITS]
n_comps = pd.qcut(n_comps, 3, labels=False)  # map to three categories (5..7)
n_comps = {i: 5 + n_comps[i] for i in DIGITS}

### Erklärung: Warum verschiedene Anzahl von Zuständen?

**Das Problem:** Verschiedene Ziffern haben unterschiedliche Aussprache-Längen:
- "1" (eins) ist kurz
- "7" (sieben) ist lang

**Die Lösung:** HMMs mit unterschiedlich vielen Zuständen anpassen

**Beispiel-Rechnung:**
```python
# Angenommen, wir haben folgende durchschnittliche Längen:
# Ziffer 0: 45 Frames
# Ziffer 1: 25 Frames  
# Ziffer 2: 35 Frames
# Ziffer 3: 30 Frames
# Ziffer 4: 40 Frames
# Ziffer 5: 50 Frames
# Ziffer 6: 38 Frames
# Ziffer 7: 55 Frames
# Ziffer 8: 42 Frames
# Ziffer 9: 48 Frames

# pd.qcut teilt in 3 Gruppen:
# Gruppe 0 (kurz): 25, 30, 35 → 5 Zustände
# Gruppe 1 (mittel): 38, 40, 42, 45 → 6 Zustände  
# Gruppe 2 (lang): 48, 50, 55 → 7 Zustände
```

**Warum ist das sinnvoll?**
- Kurze Wörter brauchen weniger Zustände (5)
- Lange Wörter brauchen mehr Zustände (7)
- So kann jedes HMM die zeitliche Struktur optimal erfassen

In [14]:
# Lass uns die Berechnung Schritt für Schritt nachvollziehen:

print("Schritt 1: Durchschnittliche Längen pro Ziffer")
print("=" * 50)

# Berechne durchschnittliche Längen für jede Ziffer
avg_lengths = []
for digit in DIGITS:
    avg_len = dataframe[dataframe["digit"] == digit].seq_length.mean()
    avg_lengths.append(avg_len)
    print(f"Ziffer {digit}: {avg_len:.1f} Frames")

print("\nSchritt 2: Sortierung nach Länge")
print("=" * 50)
sorted_digits = sorted(zip(DIGITS, avg_lengths), key=lambda x: x[1])
for digit, length in sorted_digits:
    print(f"Ziffer {digit}: {length:.1f} Frames")

print("\nSchritt 3: Quantil-Einteilung in 3 Gruppen")
print("=" * 50)
quantiles = pd.qcut(avg_lengths, 3, labels=False)
for i, (digit, quantile) in enumerate(zip(DIGITS, quantiles)):
    states = 5 + quantile
    print(f"Ziffer {digit}: Quantil {quantile} → {states} Zustände")

print("\nFinales Ergebnis:")
print("=" * 50)
print(f"n_comps = {n_comps}")

Schritt 1: Durchschnittliche Längen pro Ziffer
Ziffer 0: 50.9 Frames
Ziffer 1: 41.0 Frames
Ziffer 2: 38.9 Frames
Ziffer 3: 40.5 Frames
Ziffer 4: 41.4 Frames
Ziffer 5: 45.8 Frames
Ziffer 6: 45.4 Frames
Ziffer 7: 46.7 Frames
Ziffer 8: 41.8 Frames
Ziffer 9: 50.1 Frames

Schritt 2: Sortierung nach Länge
Ziffer 2: 38.9 Frames
Ziffer 3: 40.5 Frames
Ziffer 1: 41.0 Frames
Ziffer 4: 41.4 Frames
Ziffer 8: 41.8 Frames
Ziffer 6: 45.4 Frames
Ziffer 5: 45.8 Frames
Ziffer 7: 46.7 Frames
Ziffer 9: 50.1 Frames
Ziffer 0: 50.9 Frames

Schritt 3: Quantil-Einteilung in 3 Gruppen
Ziffer 0: Quantil 2 → 7 Zustände
Ziffer 1: Quantil 0 → 5 Zustände
Ziffer 2: Quantil 0 → 5 Zustände
Ziffer 3: Quantil 0 → 5 Zustände
Ziffer 4: Quantil 0 → 5 Zustände
Ziffer 5: Quantil 1 → 6 Zustände
Ziffer 6: Quantil 1 → 6 Zustände
Ziffer 7: Quantil 2 → 7 Zustände
Ziffer 8: Quantil 1 → 6 Zustände
Ziffer 9: Quantil 2 → 7 Zustände

Finales Ergebnis:
n_comps = {0: np.int64(7), 1: np.int64(5), 2: np.int64(5), 3: np.int64(5), 4: np.int64

### Warum Quantile? - Der Unterschied zwischen direkter Zuordnung und Quantilen

**Problem mit direkter Zuordnung:**
Wenn wir die Längen direkt in Zustände umwandeln würden, bekämen wir zu viele verschiedene Zustandsanzahlen:
- 25 Frames → 25 Zustände 
- 30 Frames → 30 Zustände
- 35 Frames → 35 Zustände
- usw.

**Das wäre schlecht, weil:**
1. **Zu viele verschiedene Modell-Größen** - schwer zu handhaben
2. **Zu komplexe Modelle** - 25+ Zustände für ein einzelnes Wort ist viel zu viel
3. **Schlechte Generalisierung** - zu spezifisch für die Trainingsdaten

**Quantile lösen das Problem:**
- **Nur 3 Kategorien** statt 10+ verschiedene Werte
- **Sinnvolle Zustandsanzahl** (5-7 Zustände) basierend auf HMM-Praxis
- **Relative Einteilung** - kurze, mittlere, lange Wörter werden fair verteilt

**Beispiel:**
Statt: `[25, 30, 35, 38, 40, 42, 45, 48, 50, 55]` Zustände
Haben wir: `[5, 5, 5, 6, 6, 6, 6, 7, 7, 7]` Zustände

Die Quantile sorgen dafür, dass etwa gleich viele Ziffern in jede Kategorie kommen!

In [15]:
# Lass uns den Unterschied zwischen direkter Zuordnung und Quantilen zeigen:

print("🔍 VERGLEICH: Direkte Zuordnung vs. Quantile")
print("=" * 60)

# Berechne durchschnittliche Längen
avg_lengths = []
for digit in DIGITS:
    avg_len = dataframe[dataframe["digit"] == digit].seq_length.mean()
    avg_lengths.append(avg_len)

print("\n1️⃣ DIREKTE ZUORDNUNG (schlecht):")
print("-" * 40)
for i, (digit, length) in enumerate(zip(DIGITS, avg_lengths)):
    # Direkte Zuordnung: Länge/5 als Zustände (nur als Beispiel)
    direct_states = max(3, int(length / 5))
    print(f"Ziffer {digit}: {length:.1f} Frames → {direct_states} Zustände")

print(f"\n❌ Problem: {len(set([max(3, int(l/5)) for l in avg_lengths]))} verschiedene Zustandsanzahlen!")

print("\n2️⃣ QUANTILE-ANSATZ (gut):")
print("-" * 40)
quantiles = pd.qcut(avg_lengths, 3, labels=False)
for i, (digit, length, quantile) in enumerate(zip(DIGITS, avg_lengths, quantiles)):
    states = 5 + quantile
    print(f"Ziffer {digit}: {length:.1f} Frames → Quantil {quantile} → {states} Zustände")

print(f"\n✅ Vorteil: Nur 3 verschiedene Zustandsanzahlen (5, 6, 7)!")

# Zeige die gleichmäßige Verteilung
print("\n3️⃣ GLEICHMÄSSIGE VERTEILUNG:")
print("-" * 40)
for i in range(3):
    count = sum(1 for q in quantiles if q == i)
    print(f"Quantil {i} ({5+i} Zustände): {count} Ziffern")

print(f"\n💡 Das ist der Sinn der Quantile: Gleichmäßige Verteilung statt willkürliche Werte!")

🔍 VERGLEICH: Direkte Zuordnung vs. Quantile

1️⃣ DIREKTE ZUORDNUNG (schlecht):
----------------------------------------
Ziffer 0: 50.9 Frames → 10 Zustände
Ziffer 1: 41.0 Frames → 8 Zustände
Ziffer 2: 38.9 Frames → 7 Zustände
Ziffer 3: 40.5 Frames → 8 Zustände
Ziffer 4: 41.4 Frames → 8 Zustände
Ziffer 5: 45.8 Frames → 9 Zustände
Ziffer 6: 45.4 Frames → 9 Zustände
Ziffer 7: 46.7 Frames → 9 Zustände
Ziffer 8: 41.8 Frames → 8 Zustände
Ziffer 9: 50.1 Frames → 10 Zustände

❌ Problem: 4 verschiedene Zustandsanzahlen!

2️⃣ QUANTILE-ANSATZ (gut):
----------------------------------------
Ziffer 0: 50.9 Frames → Quantil 2 → 7 Zustände
Ziffer 1: 41.0 Frames → Quantil 0 → 5 Zustände
Ziffer 2: 38.9 Frames → Quantil 0 → 5 Zustände
Ziffer 3: 40.5 Frames → Quantil 0 → 5 Zustände
Ziffer 4: 41.4 Frames → Quantil 0 → 5 Zustände
Ziffer 5: 45.8 Frames → Quantil 1 → 6 Zustände
Ziffer 6: 45.4 Frames → Quantil 1 → 6 Zustände
Ziffer 7: 46.7 Frames → Quantil 2 → 7 Zustände
Ziffer 8: 41.8 Frames → Quantil 1 → 6 

In [6]:
### TODO: 
### 1. implement the 6-fold cross-validation loop
### 2. allocate and initialize the HMMs, one for each digit; set a linear topology
### 3. train the HMMs using the fit method; data needs to be concatenated
### 4. evaluate the trained models on the test speaker; how do you decide which word
###    was spoken?

from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

y_true_per_speaker = {}
y_pred_per_speaker = {}

model_folds = {}

for ts in SPEAKERS:
    df_train = dataframe[dataframe["speaker"] != ts]
    df_test = dataframe.drop(df_train.index)

    # allocate and initialize the HMMs, one for each digit
    # choose and a meaningful number of states
    models = {i: hmm.GaussianHMM(n_components=n_comps[i], covariance_type='diag', init_params='c', params='tmc', n_iter=10, verbose=False) for i in DIGITS}

    for d, m in sorted(models.items()):
        print(f"Training model for {d}: {m}")

        # make linear topology
        m.startprob_ = np.zeros(m.n_components); m.startprob_[0] = 1

        m.transmat_ = np.zeros([m.n_components, m.n_components])
        duration = 0.8
        for i in range(0, m.n_components-1):
            m.transmat_[i, i] = duration
            m.transmat_[i, i+1] = 1.0 - duration
        m.transmat_[-1, -1] = 1

        # work on the training subset for the current digit
        df_train_d = df_train[dataframe["digit"] == d]
        
        # initialize the means by making forced linear alignments over the data
        # ie, splitting the sequences evenly into n_comp parts and add those to
        # the respective bins
        fla = [[] for i in range(0, m.n_components)]
        for o in df_train_d.features:
            for i, slice in enumerate(np.array_split(o, m.n_components)):
                fla[i].append(slice)
        m.means_ = np.array([np.mean(np.concatenate(x), axis=0) for x in fla])
        
        # train the HMMs using the fit method; data needs to be concatenated,
        # see https://github.com/hmmlearn/hmmlearn/blob/38b3cece4a6297e978a204099ae6a0a99555ec01/lib/hmmlearn/base.py#L439
        
        X = np.concatenate([x for x in df_train_d.features])  # "unwrap" the np array
        lengths = np.array([len(x) for x in df_train_d.features])

        m.fit(X, lengths)

        print(f'Training of {d} completed: {m.monitor_}')
    
    print('Evaluating on training set...')
    yt_true = []
    yt_pred = []
    for i, x in df_train.iterrows():
        yt_true.append(int(x.digit))
        yt_pred.append(np.argmax([models[j].score(x.features) for j in DIGITS]))
    
    print(confusion_matrix(yt_true, yt_pred, labels=DIGITS))
    print(classification_report(yt_true, yt_pred, labels=DIGITS))

    # evaluate the trained models on the test speaker; how do you decide which word
    # was spoken?
    print('Evaluating on test set (held-out speaker)...')
    yt_true = []
    yt_pred = []
    for i, x in df_test.iterrows():
        yt_true.append(int(x.digit))
        yt_pred.append(np.argmax([models[j].score(x.features) for j in DIGITS]))
    
    print(confusion_matrix(yt_true, yt_pred, labels=DIGITS))
    print(classification_report(yt_true, yt_pred, labels=DIGITS))

    y_true.extend(yt_true)
    y_pred.extend(yt_pred)

    y_true_per_speaker[ts] = yt_true
    y_pred_per_speaker[ts] = yt_pred

    # store the individual models for later use
    model_folds[ts] = models

Training model for 0: GaussianHMM(init_params='c', n_components=7, params='tmc')


/var/folders/fz/9kmbj46x27q5mz39hqvmh3x40000gn/T/ipykernel_7737/1208296208.py:40: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_train_d = df_train[dataframe["digit"] == d]


Training of 0 completed: ConvergenceMonitor(
    history=[-204800.64728771578, -186499.77093047713, -184870.76720162595, -184534.1553442875, -183978.96196567727, -183666.53269133085, -183591.40109485175, -183532.96804679552, -183493.14300435077, -183476.78435418877],
    iter=10,
    n_iter=10,
    tol=0.01,
    verbose=False,
)
Training model for 1: GaussianHMM(init_params='c', n_components=5, params='tmc')
Training of 1 completed: ConvergenceMonitor(
    history=[-162453.91319780136, -149301.27002669274, -148981.45079784672, -148924.87096246495, -148889.8275925376, -148863.37648346668, -148842.57582746202, -148825.8058914809, -148811.98430020708, -148800.36093756964],
    iter=10,
    n_iter=10,
    tol=0.01,
    verbose=False,
)
Training model for 2: GaussianHMM(init_params='c', n_components=5, params='tmc')


/var/folders/fz/9kmbj46x27q5mz39hqvmh3x40000gn/T/ipykernel_7737/1208296208.py:40: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_train_d = df_train[dataframe["digit"] == d]
/var/folders/fz/9kmbj46x27q5mz39hqvmh3x40000gn/T/ipykernel_7737/1208296208.py:40: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_train_d = df_train[dataframe["digit"] == d]


Training of 2 completed: ConvergenceMonitor(
    history=[-160260.67445958278, -146870.50915755524, -146210.71432917894, -146118.33180730333, -146029.63593956284, -145947.58061781042, -145880.71058220265, -145825.10836516516, -145779.81772360977, -145744.92027308064],
    iter=10,
    n_iter=10,
    tol=0.01,
    verbose=False,
)
Training model for 3: GaussianHMM(init_params='c', n_components=5, params='tmc')
Training of 3 completed: ConvergenceMonitor(
    history=[-166569.15339253895, -152379.65114060807, -151896.38138141166, -151768.7164227543, -151663.2565943423, -151554.06064248664, -151423.371691292, -151310.80433419006, -151224.13071452457, -151161.76150000386],
    iter=10,
    n_iter=10,
    tol=0.01,
    verbose=False,
)
Training model for 4: GaussianHMM(init_params='c', n_components=5, params='tmc')


...

Training of 7 completed: ConvergenceMonitor(
    history=[-183341.36579094568, -166837.282153623, -165510.76962097076, -165218.71568157733, -165051.5258447243, -164904.26663346728, -164703.5831950163, -164598.05526593258, -164564.85147759755, -164548.7697919545],
    iter=10,
    n_iter=10,
    tol=0.01,
    verbose=False,
)
Training model for 8: GaussianHMM(init_params='c', n_components=6, params='tmc')


/var/folders/fz/9kmbj46x27q5mz39hqvmh3x40000gn/T/ipykernel_7737/1208296208.py:40: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_train_d = df_train[dataframe["digit"] == d]


Training of 8 completed: ConvergenceMonitor(
    history=[-169500.30249000527, -156296.32444782078, -155089.85242623402, -154891.8528596493, -154842.75471836535, -154818.17655667078, -154803.31934682906, -154793.19979482456, -154786.12213435565, -154780.51664468943],
    iter=10,
    n_iter=10,
    tol=0.01,
    verbose=False,
)
Training model for 9: GaussianHMM(init_params='c', n_components=7, params='tmc')


/var/folders/fz/9kmbj46x27q5mz39hqvmh3x40000gn/T/ipykernel_7737/1208296208.py:40: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_train_d = df_train[dataframe["digit"] == d]


Training of 9 completed: ConvergenceMonitor(
    history=[-206372.5071199321, -188337.185815717, -187628.98894830406, -187441.69421249678, -187334.69324531266, -187216.7174518333, -186876.09456341868, -186176.3511845874, -185683.44739178146, -185570.28205325617],
    iter=10,
    n_iter=10,
    tol=0.01,
    verbose=False,
)
Evaluating on training set...
[[248   0   1   1   0   0   0   0   0   0]
 [  0 240   0   0   0   9   0   1   0   0]
 [  6   0 244   0   0   0   0   0   0   0]
 [  1   0   2 242   0   0   2   1   2   0]
 [  0   6   0   0 244   0   0   0   0   0]
 [  0   2   0   0   0 247   0   0   0   1]
 [  0   0   1   6   0   1 217   0  25   0]
 [  0   0   0   2   0   0   4 244   0   0]
 [  1   0   0   0   0   0  10   0 239   0]
 [  0   0   0   0   0   0   0   0   0 250]]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       250
           1       0.97      0.96      0.96       250
           2       0.98      0.98      0.98      

### Erklärung: HMM-Training und Cross-Validation

Dieser Code-Block implementiert die **6-fold Cross-Validation** für die HMM-basierte Spracherkennung.

#### **1. Cross-Validation Setup**
```python
for ts in SPEAKERS:  # Jeder Speaker wird einmal als Testset verwendet
    df_train = dataframe[dataframe["speaker"] != ts]  # 5 Speaker für Training
    df_test = dataframe.drop(df_train.index)          # 1 Speaker für Test
```
- **6-fold**: 6 Durchläufe, jeder Speaker wird einmal als Testset verwendet
- **Speaker-unabhängig**: Trainiere mit 5 Sprechern, teste mit 1 Sprecher

#### **2. HMM-Initialisierung**
```python
models = {i: hmm.GaussianHMM(n_components=n_comps[i], ...) for i in DIGITS}
```
- **10 separate HMMs**: Ein Modell pro Ziffer (0-9)
- **Verschiedene Zustandsanzahlen**: Aus der Quantil-Berechnung

#### **3. Lineare Topologie**
```python
m.startprob_[0] = 1                    # Starte immer bei Zustand 0
m.transmat_[i, i] = 0.8               # 80% Chance im gleichen Zustand bleiben
m.transmat_[i, i+1] = 0.2             # 20% Chance zum nächsten Zustand
```
- **Linear**: Zustand 0 → 1 → 2 → ... → Ende
- **Keine Sprünge**: Kann nur vorwärts oder bleiben

#### **4. Training**
```python
X = np.concatenate([x for x in df_train_d.features])  # Alle Features zusammenfügen
lengths = np.array([len(x) for x in df_train_d.features])  # Längen der Sequenzen
m.fit(X, lengths)  # HMM trainieren
```

#### **5. Evaluation**
```python
yt_pred.append(np.argmax([models[j].score(x.features) for j in DIGITS]))
```
- **Teste alle 10 HMMs** auf jede Testsequenz
- **Wähle das HMM mit höchstem Score** als Vorhersage

### Visualisierung der linearen Topologie

**Beispiel für ein 5-Zustände HMM:**

```
START → [S0] → [S1] → [S2] → [S3] → [S4] → END
         ↺      ↺      ↺      ↺      ↺
        80%    80%    80%    80%   100%
```

**Übergangsmatrix (5x5):**
```
     S0   S1   S2   S3   S4
S0 [ 0.8  0.2  0.0  0.0  0.0 ]
S1 [ 0.0  0.8  0.2  0.0  0.0 ]
S2 [ 0.0  0.0  0.8  0.2  0.0 ]
S3 [ 0.0  0.0  0.0  0.8  0.2 ]
S4 [ 0.0  0.0  0.0  0.0  1.0 ]
```

### Warum lineare Topologie?

1. **Zeitliche Struktur**: Sprache ist zeitlich sequenziell
2. **Natürliche Progression**: Wörter haben Anfang, Mitte, Ende
3. **Verhindert Zyklen**: Keine Rücksprünge in der Zeit
4. **Einfach zu verstehen**: Jeder Zustand repräsentiert eine Phase des Wortes

### Training-Prozess im Detail

**Schritt 1: Datenaufbereitung**
- Alle Trainingssequenzen einer Ziffer werden zusammengefügt
- `lengths` Array speichert die ursprünglichen Sequenzlängen

**Schritt 2: Initialisierung**
- **Startwahrscheinlichkeiten**: Nur Zustand 0 kann starten
- **Mittelwerte**: Durch gleichmäßige Aufteilung der Daten initialisiert
- **Übergangsmatrix**: Lineare Topologie mit 80% Verweildauer

**Schritt 3: EM-Algorithmus**
- **E-Schritt**: Berechne die wahrscheinlichsten Zustandssequenzen
- **M-Schritt**: Update die Parameter (Mittelwerte, Kovarianzen, Übergänge)
- **Wiederholung**: Bis Konvergenz oder max. Iterationen

### Wo sind die Emissionswahrscheinlichkeiten?

**Antwort: Sie werden automatisch erstellt und trainiert!**

#### **1. Was sind Emissionswahrscheinlichkeiten?**
- **P(Beobachtung | Zustand)**: Wahrscheinlichkeit, dass ein Zustand bestimmte MFCC-Features "ausstößt"
- **Gaußsche Verteilung**: Jeder Zustand hat eine 13-dimensionale Gaußverteilung (für 13 MFCC-Features)
- **Parameter**: Mittelwerte (`means_`) und Kovarianzen (`covars_`)

#### **2. Wo werden sie erstellt?**
```python
# Automatisch bei der HMM-Initialisierung:
hmm.GaussianHMM(n_components=5, covariance_type='diag', ...)

# Nach dem Training enthalten:
m.means_   # Mittelwerte der Gaußverteilungen (5 x 13)
m.covars_  # Kovarianzen der Gaußverteilungen (5 x 13)
```

#### **3. Initialisierung der Emissionen**
```python
# Intelligente Initialisierung durch "Forced Linear Alignment":
fla = [[] for i in range(0, m.n_components)]
for o in df_train_d.features:
    for i, slice in enumerate(np.array_split(o, m.n_components)):
        fla[i].append(slice)
m.means_ = np.array([np.mean(np.concatenate(x), axis=0) for x in fla])
```
- **Sequenzen werden gleichmäßig aufgeteilt** in n_components Teile
- **Jeder Zustand bekommt** seine entsprechenden MFCC-Features
- **Mittelwerte werden berechnet** für jeden Zustand

#### **4. Training der Emissionen**
```python
m.fit(X, lengths)  # EM-Algorithmus optimiert automatisch:
```
- **Mittelwerte** (`means_`) werden angepasst
- **Kovarianzen** (`covars_`) werden angepasst
- **Übergangswahrscheinlichkeiten** werden angepasst

In [ ]:
# Lass uns die Emissionswahrscheinlichkeiten visualisieren!
# (Diesen Code NACH dem Training ausführen)

def show_emission_parameters(model, digit):
    """Zeigt die Emissionsparameter eines trainierten HMMs"""
    print(f"\n🔍 EMISSIONSPARAMETER für Ziffer {digit}")
    print("=" * 50)
    
    n_states = model.n_components
    n_features = model.n_features
    
    print(f"Anzahl Zustände: {n_states}")
    print(f"Anzahl Features: {n_features} (MFCC-Koeffizienten)")
    print(f"Kovarianztyp: {model.covariance_type}")
    
    print(f"\n📊 MITTELWERTE (means_):")
    print(f"Shape: {model.means_.shape}")
    for state in range(n_states):
        print(f"Zustand {state}: {model.means_[state][:5]}... (erste 5 MFCC-Koeffs)")
    
    print(f"\n📈 KOVARIANZEN (covars_):")
    print(f"Shape: {model.covars_.shape}")
    for state in range(n_states):
        print(f"Zustand {state}: {model.covars_[state][:5]}... (erste 5 Varianzen)")
    
    return model.means_, model.covars_

# Beispiel: Zeige Parameter für die erste Ziffer (nach dem Training)
print("⚠️  Diesen Code NACH dem Training ausführen!")
print("Beispiel für die Verwendung:")
print("show_emission_parameters(models[0], 0)  # Für Ziffer 0")

In [7]:
### TODO: 
### 1. based on the results, compute and display the confusion matrix for 
###    each test speaker 
### 2. compute and display the confusion matrix for the overall dataset

print("Results per speaker:")
for speaker in SPEAKERS:
    print("### {}".format(speaker))

    yt_true = y_true_per_speaker[speaker]
    yt_pred = y_pred_per_speaker[speaker]
    
    print(confusion_matrix(yt_true, yt_pred, labels=DIGITS))
    print(classification_report(yt_true, yt_pred, labels=DIGITS))
    print()

print("Overall result after cross-validation:")
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

Results per speaker:
### george
[[50  0  0  0  0  0  0  0  0  0]
 [ 0 48  0  0  0  1  0  1  0  0]
 [20 21  0  0  7  0  0  0  0  2]
 [ 0  0  0 46  0  0  2  0  2  0]
 [ 0 20  0  0 30  0  0  0  0  0]
 [ 0  0  0  1  0 45  0  0  0  4]
 [ 0  0  0 13  0  0 28  0  9  0]
 [ 0  0  0  0  0  0  0 50  0  0]
 [ 0  0  0  0  0  0  0  0 50  0]
 [ 0  0  0  0  0  0  0  0  0 50]]
              precision    recall  f1-score   support

           0       0.71      1.00      0.83        50
           1       0.54      0.96      0.69        50
           2       0.00      0.00      0.00        50
           3       0.77      0.92      0.84        50
           4       0.81      0.60      0.69        50
           5       0.98      0.90      0.94        50
           6       0.93      0.56      0.70        50
           7       0.98      1.00      0.99        50
           8       0.82      1.00      0.90        50
           9       0.89      1.00      0.94        50

    accuracy                           0.

/Users/seebergerph/anaconda3/envs/seqlrn/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/seebergerph/anaconda3/envs/seqlrn/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/seebergerph/anaconda3/envs/seqlrn/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

---

## Task 2) Decoding Sequences of Digits

The example above can't handle sequences of spoken digits.
In this part of the assignment, you'll build a basic decoder that is able to decode arbitrary sequences of digits (without a prior, though).
The `decode` method in `hmmlearn` only works for a single HMM.
There are two ways how to solve this assignment:

- Construct a "meta" HMM from the previously trained digit HMMs, by allowing state transitions from one digit to another; the resulting HMM can be decoded using the existing `decode` method (don't forget to re-map the state ids to the originating digit).

- (Optional) Implement a real (time-synchronous) decoder using beam search. The straight-forward way is to maintain a (sorted) list of active hypotheses (ie. state history and current log-likelihood) that is first expanded and then pruned in each time step. The tricky part is at the "end" of a model: do you loop or expand new words?

---

### Generate Test Sequences

3.1 Generate a few test sequences of random length in between 3 and 6 digits; use [`numpy.random.randint`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randint.html) and be sure to also retain the digits sequence since we need to compute edit distance between reference and hypotheses later.

In [8]:
def create_digit_sequence(speaker_dataframe, min_digits=3, max_digits=7):
    """
    Creates a sequence of spoken digits from a speaker and returns the
    features and reference label.
    """
    test_seq = speaker_dataframe.sample(n=np.random.randint(min_digits, max_digits))
    digits = " ".join(map(str, test_seq.digit))
    return test_seq, digits

In [9]:
### Notice: just for test purposes
speaker = "george"

data_george = dataframe[dataframe["speaker"] == speaker]
for i in range(20):
    data_seq, digits = create_digit_sequence(data_george)
    print("Digits: {}".format(digits))

Digits: 7 1 6 3 8
Digits: 0 1 2
Digits: 2 0 6 9 3
Digits: 6 6 1 4
Digits: 8 5 0 1 7
Digits: 4 6 6
Digits: 4 6 1 7 2
Digits: 3 5 0 8 0
Digits: 0 2 6 1
Digits: 7 6 9 8 3
Digits: 4 9 4 9
Digits: 0 4 8 4
Digits: 1 5 7 6
Digits: 8 8 0 8
Digits: 3 8 1
Digits: 4 3 8 3 1 7
Digits: 7 9 8 2 5 5
Digits: 6 2 5 7 9
Digits: 8 3 9
Digits: 0 8 5 3


### Create "meta" HMM

4.1 Combine the previously trained HMMs to a single "meta" HMM, altering the transition probabilities to make a circular graph that allows each word to follow another.

4.2 Implement a method that converts a state sequence relating to the meta HMM into a sequence of actual digits.

4.3 Decode your test sequences and compute the [word error rate](https://en.wikipedia.org/wiki/Word_error_rate) (WER) with [JiWER](https://pypi.org/project/jiwer/) (install the package in your working environment).

4.4 Compute an overall WER; ie. over the cross-validation.

4.5 (Optional) Implement a basic time-synchronous beam search; how do the results compare to the above viterbi decoding in terms of accuracy and time?

In [10]:
hyps = []
refs = []

wers = []
for ts in SPEAKERS:
    # restore the model for this fold
    models = model_folds[ts]

    # combine the (previously trained) per-digit HMMs into one large meta HMM; make
    # sure to change the transition probabilities to allow transitions from one
    # digit to any other; use full since after fit() the hmms all have non-sparse diags
    m = hmm.GaussianHMM(n_components=sum(n_comps.values()), covariance_type='full', init_params='', params='mc', n_iter=10, verbose=False)

    # copy dim, means and covars
    m.n_features = 13
    m.means_ = np.concatenate([i.means_ for _, i in sorted(models.items())])
    m.covars_ = np.concatenate([i.covars_ for _, i in sorted(models.items())])

    # set equal prior on entry
    offsets = [0]
    for i, n in sorted(n_comps.items()):
        offsets.append(offsets[-1] + n)
    offsets = np.array(offsets)

    # startprops with equal priors
    m.startprob_ = np.zeros(m.n_components)
    m.startprob_[offsets[:-1]] = 1. / len(DIGITS)

    # transitions: copy and adjust last state
    m.transmat_ = np.zeros([m.n_components, m.n_components])
    for d, dm in sorted(models.items()):
        c = dm.n_components
        o = offsets[d]

        # block-copy the original transmat
        m.transmat_[o:o+c, o:o+c] = dm.transmat_
        
        # at the last state, allow equal transition to other digits
        remain = 0.7
        m.transmat_[o+c-1, offsets[:-1]] = (1.0 - remain) / len(DIGITS)
        m.transmat_[o+c-1, o+c-1] = remain

    # test

    # for 20 times, randomly sample between 3 and 6 words from the testset
    shyps = []
    srefs = []
    for i in range(0, 20):
        speaker_data = dataframe[dataframe['speaker'] == ts]
        test_seq, ref = create_digit_sequence(speaker_data)
        #ref = " ".join(ref)

        # concat features
        seq = np.concatenate([x for x in test_seq.features])

        # use the `decode` function to get the most likely state sequence for the test
        # sequences; re-map that to a sequence of digits
        logit, states = m.decode(seq)
        initial_states = [k for k, g in groupby(states) if k in offsets]
        hyp = " ".join(map(str, np.digitize(initial_states, offsets, right=True)))

        # use jiwer.wer to compute the word error rate between reference and decoded
        # digit sequence
        print(f'{wer(ref, hyp):.2f} ref="{ref}" hyp="{hyp}"')

        srefs.append(ref)
        shyps.append(hyp)

    refs.extend(srefs)
    hyps.extend(shyps)

    wers.append((ts, wer(shyps, srefs)))

for s, e in wers:
    print(f'WER {s} = {e:.2f}')

print(f'WER overall = {wer(hyps, refs):.2f}')

0.67 ref="7 5 3 2 2 6" hyp="7 5 2 3 0 0 3 6"
0.17 ref="8 8 3 3 2 8" hyp="8 8 3 3 4 8"
0.67 ref="2 1 6" hyp="1 3 6"
0.67 ref="9 3 3" hyp="9 7 3 3 9"
0.00 ref="1 7 1" hyp="1 7 1"
0.67 ref="0 2 2" hyp="0 4 1"
0.25 ref="9 2 1 9" hyp="9 1 9"
0.60 ref="7 7 6 4 5" hyp="6 7 7 6 1 5 2"
0.20 ref="7 7 7 8 0" hyp="7 7 7 8 0 1"
0.40 ref="2 9 4 2 0" hyp="1 9 4 2 0 0"
0.20 ref="0 7 6 0 6" hyp="0 7 3 0 6"
0.50 ref="3 2 0 8 6 6" hyp="7 3 4 0 8 3 6"
0.67 ref="6 4 2 2 4 0" hyp="6 1 0 0 1 0"
0.75 ref="4 5 2 8" hyp="4 1 5 8 0 8"
0.00 ref="1 7 8" hyp="1 7 8"
0.67 ref="4 6 7 2 0 2" hyp="4 1 6 7 0 0 0 9"
0.67 ref="2 0 5 1 7 2" hyp="0 0 5 2 1 7 0 9"
0.20 ref="5 8 7 3 3" hyp="5 8 7 8 3"
0.00 ref="7 9 8" hyp="7 9 8"
0.00 ref="0 8 3" hyp="0 8 3"
0.33 ref="7 8 7" hyp="7 8 5"
0.00 ref="0 0 6 8" hyp="0 0 6 8"
0.20 ref="4 5 5 9 0" hyp="1 5 5 9 0"
0.20 ref="2 4 9 1 3" hyp="2 1 9 1 3"
0.25 ref="0 2 7 4" hyp="0 2 7 4 5"
0.20 ref="8 4 1 7 1" hyp="8 4 1 7 1 9"
0.00 ref="7 7 7 9 3 0" hyp="7 7 7 9 3 0"
0.00 ref="5 1 3" hyp=

### Erklärung: Meta-HMM für Sequenzerkennung

Dieser Code erstellt ein **"Meta-HMM"** aus den einzelnen Ziffer-HMMs, um **Sequenzen von Ziffern** zu erkennen (z.B. "1 2 3" statt nur einzelne Ziffern).

#### **🎯 Das Problem:**
- Einzelne HMMs können nur **eine Ziffer** erkennen
- Für Sequenzen wie "1 2 3" brauchen wir ein **kombiniertes Modell**
- Das Meta-HMM erlaubt **Übergänge zwischen Ziffern**

#### **🔧 Die Lösung: Meta-HMM Construction**

**Schritt 1: Große HMM erstellen**
```python
m = hmm.GaussianHMM(n_components=sum(n_comps.values()), ...)
```
- **Alle Zustände kombinieren**: Ziffer 0 (5 Zustände) + Ziffer 1 (6 Zustände) + ... = ~60 Zustände
- **Ein großes HMM** statt 10 kleine HMMs

**Schritt 2: Parameter kopieren**
```python
m.means_ = np.concatenate([i.means_ for _, i in sorted(models.items())])
m.covars_ = np.concatenate([i.covars_ for _, i in sorted(models.items())])
```
- **Emissionsparameter**: Kopiere alle Mittelwerte und Kovarianzen
- **Reihenfolge**: Ziffer 0, dann Ziffer 1, dann Ziffer 2, usw.

**Schritt 3: Offset-Berechnung**
```python
offsets = [0, 5, 11, 17, ...]  # Startpositionen der Ziffern
```
- **Ziffer 0**: Zustände 0-4
- **Ziffer 1**: Zustände 5-10  
- **Ziffer 2**: Zustände 11-16
- usw.

**Schritt 4: Startwahrscheinlichkeiten**
```python
m.startprob_[offsets[:-1]] = 1. / len(DIGITS)
```
- **Jede Ziffer kann starten**: Gleiche Wahrscheinlichkeit für alle ersten Zustände
- **Nur erste Zustände**: 0, 5, 11, 17, ... können starten

**Schritt 5: Übergangsmatrix**
```python
# Kopiere interne Übergänge
m.transmat_[o:o+c, o:o+c] = dm.transmat_

# Übergänge zwischen Ziffern
m.transmat_[o+c-1, offsets[:-1]] = (1.0 - remain) / len(DIGITS)
```
- **Innerhalb einer Ziffer**: Behalte die lineare Topologie
- **Zwischen Ziffern**: Vom letzten Zustand einer Ziffer kann man zu jeder anderen Ziffer springen

### Visualisierung des Meta-HMM

**Beispiel: 3 Ziffern mit je 3 Zuständen**

```
Ziffer 0:  [S0] → [S1] → [S2] ↘
                               ↘
Ziffer 1:          [S3] → [S4] → [S5] ↘
                                      ↘
Ziffer 2:                  [S6] → [S7] → [S8]
           ↑                ↑              ↑
           START           START          START
```

**Übergangsmatrix Struktur:**
```
      S0  S1  S2  S3  S4  S5  S6  S7  S8
S0  [ 0.8 0.2 0.0 0.0 0.0 0.0 0.0 0.0 0.0 ]
S1  [ 0.0 0.8 0.2 0.0 0.0 0.0 0.0 0.0 0.0 ]
S2  [ 0.1 0.0 0.7 0.1 0.0 0.0 0.1 0.0 0.0 ]  ← Übergänge!
S3  [ 0.0 0.0 0.0 0.8 0.2 0.0 0.0 0.0 0.0 ]
S4  [ 0.0 0.0 0.0 0.0 0.8 0.2 0.0 0.0 0.0 ]
S5  [ 0.1 0.0 0.0 0.1 0.0 0.7 0.1 0.0 0.0 ]  ← Übergänge!
S6  [ 0.0 0.0 0.0 0.0 0.0 0.0 0.8 0.2 0.0 ]
S7  [ 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.8 0.2 ]
S8  [ 0.1 0.0 0.0 0.1 0.0 0.0 0.1 0.0 0.7 ]  ← Übergänge!
```

### Dekodierung-Prozess

**Schritt 1: Viterbi-Dekodierung**
```python
logit, states = m.decode(seq)
```
- **Input**: Komplette MFCC-Sequenz (z.B. "1 2 3")
- **Output**: Wahrscheinlichste Zustandssequenz (z.B. [0,1,2,3,4,5,6,7,8])

**Schritt 2: Zustandssequenz → Ziffernsequenz**
```python
initial_states = [k for k, g in groupby(states) if k in offsets]
hyp = " ".join(map(str, np.digitize(initial_states, offsets, right=True)))
```
- **Gruppierung**: Finde nur die "Start-Zustände" (0, 3, 6, ...)
- **Mapping**: Zustand → Ziffer (0→0, 3→1, 6→2)
- **Ergebnis**: "0 1 2"

### Warum funktioniert das?

1. **Zeitliche Kontinuität**: Das Meta-HMM kann beliebig lange Sequenzen verarbeiten
2. **Wortgrenzen**: Übergänge zwischen Ziffern ermöglichen Sequenzerkennung
3. **Viterbi-Algorithmus**: Findet den optimalen Pfad durch das gesamte Meta-HMM
4. **Zustandsmapping**: Konvertiert Zustandssequenz zurück zu Ziffernsequenz

### Limitationen

- **Keine Wortgrenzen-Erkennung**: Schwierig zu erkennen, wo ein Wort endet
- **Gleichverteilte Übergänge**: Alle Ziffern sind gleich wahrscheinlich
- **Keine Sprachmodellierung**: Keine Berücksichtigung von Ziffernfolgen-Wahrscheinlichkeiten

## Der Viterbi-Algorithmus - Schritt für Schritt erklärt

### 🎯 Was ist der Viterbi-Algorithmus?

Der **Viterbi-Algorithmus** findet die **wahrscheinlichste Zustandssequenz** in einem HMM für gegebene Beobachtungen.

**Problem**: Gegeben MFCC-Features, welche Zustandssequenz ist am wahrscheinlichsten?

### 🔧 Wie funktioniert er?

**Grundprinzip**: **Dynamische Programmierung** - löse das Problem schrittweise von links nach rechts.

#### **Input**:
- **Beobachtungen**: O = [o₁, o₂, o₃, ..., oₜ] (MFCC-Features)
- **HMM-Parameter**: Startwahrscheinlichkeiten, Übergangswahrscheinlichkeiten, Emissionswahrscheinlichkeiten

#### **Output**:
- **Beste Zustandssequenz**: S = [s₁, s₂, s₃, ..., sₜ]
- **Wahrscheinlichkeit**: P(S|O)

### 📊 Algorithmus-Schritte

#### **Schritt 1: Initialisierung (t=1)**
```python
# Für jeden Zustand i:
V[1][i] = π[i] * B[i][o₁]
path[1][i] = i
```
- **V[t][i]**: Beste Wahrscheinlichkeit für Zustand i zum Zeitpunkt t
- **π[i]**: Startwahrscheinlichkeit für Zustand i
- **B[i][o₁]**: Emissionswahrscheinlichkeit für Beobachtung o₁ in Zustand i

#### **Schritt 2: Rekursion (t=2 bis T)**
```python
# Für jeden Zustand j zum Zeitpunkt t:
for j in states:
    # Finde den besten Vorgänger-Zustand
    best_prob = max(V[t-1][i] * A[i][j] for i in states)
    best_prev = argmax(V[t-1][i] * A[i][j] for i in states)
    
    # Aktualisiere
    V[t][j] = best_prob * B[j][oₜ]
    path[t][j] = best_prev
```
- **A[i][j]**: Übergangswahrscheinlichkeit von Zustand i zu Zustand j
- **Backtracking**: Speichere den besten Vorgänger-Zustand

#### **Schritt 3: Terminierung**
```python
# Finde den besten Endzustand
best_final_prob = max(V[T][i] for i in states)
best_final_state = argmax(V[T][i] for i in states)
```

#### **Schritt 4: Pfad-Rekonstruktion**
```python
# Rückwärts durch path[] gehen
optimal_path = []
current_state = best_final_state
for t in range(T, 0, -1):
    optimal_path.append(current_state)
    current_state = path[t][current_state]
optimal_path.reverse()
```

### 🎨 Visuelles Beispiel

**Einfaches HMM**: 3 Zustände (S0, S1, S2), 3 Zeitschritte

```
Zeitschritt:    t=1      t=2      t=3
Beobachtung:    o₁       o₂       o₃

Zustände:
S0 ────────────●────────●────────●
               │╲       │╲       │
S1 ────────────●─●──────●─●──────●
               │ │╲     │ │╲     │
S2 ────────────●─●─●────●─●─●────●
```

**Viterbi-Tabelle**:
```
       t=1    t=2    t=3
S0   [ 0.4  ] [0.12] [0.03]
S1   [ 0.3  ] [0.15] [0.06]
S2   [ 0.1  ] [0.08] [0.04]
```

**Pfad-Tabelle** (bester Vorgänger):
```
       t=1    t=2    t=3
S0   [ -   ] [ S0 ] [ S1 ]
S1   [ -   ] [ S0 ] [ S1 ]
S2   [ -   ] [ S1 ] [ S1 ]
```

**Beste Sequenz**: S0 → S1 → S1 (Wahrscheinlichkeit: 0.06)

### 💡 Intuitive Erklärung

**Stell dir vor, du gehst durch einen Garten**:
- **Zeitschritte**: Schritte vorwärts
- **Zustände**: Verschiedene Pfade (Wege)
- **Beobachtungen**: Was du siehst (Blumen, Bäume)
- **Viterbi**: Findet den besten Pfad basierend auf dem, was du siehst

**Warum funktioniert es?**
1. **Optimalitätsprinzip**: Der beste Pfad zu einem Punkt muss aus dem besten Pfad zum vorherigen Punkt bestehen
2. **Dynamische Programmierung**: Löse kleinere Teilprobleme und kombiniere sie
3. **Lokale Entscheidungen**: In jedem Zeitschritt nur die beste Option behalten

### 🔍 Mathematische Formeln

**Rekursionsformel**:
```
V[t][j] = max(V[t-1][i] * A[i][j]) * B[j][oₜ]
          i
```

**Bedeutung**:
- **V[t-1][i]**: Beste Wahrscheinlichkeit bis zum vorherigen Zeitschritt
- **A[i][j]**: Übergangswahrscheinlichkeit von Zustand i zu j
- **B[j][oₜ]**: Emissionswahrscheinlichkeit für Beobachtung oₜ in Zustand j

### ⚡ Effizienz

**Zeitkomplexität**: O(T × N²)
- T: Anzahl Zeitschritte
- N: Anzahl Zustände

**Speicherkomplexität**: O(T × N)

**Warum effizient?**
- Ohne Viterbi: N^T mögliche Sequenzen zu überprüfen
- Mit Viterbi: Nur T × N² Berechnungen nötig

In [18]:
# Einfache Viterbi-Implementierung (zur Demonstration)

def viterbi_demo(observations, start_prob, trans_prob, emit_prob, states):
    """
    Einfache Viterbi-Implementierung für Demonstrationszwecke
    
    Args:
        observations: Liste der Beobachtungen
        start_prob: Startwahrscheinlichkeiten {state: prob}
        trans_prob: Übergangswahrscheinlichkeiten {state: {state: prob}}
        emit_prob: Emissionswahrscheinlichkeiten {state: {obs: prob}}
        states: Liste der Zustände
    """
    
    T = len(observations)
    N = len(states)
    
    # Initialisiere Viterbi-Tabelle und Pfad-Tabelle
    V = [{} for _ in range(T)]
    path = [{} for _ in range(T)]
    
    print("🔍 VITERBI-ALGORITHMUS DEMONSTRATION")
    print("=" * 50)
    print(f"Beobachtungen: {observations}")
    print(f"Zustände: {states}")
    print()
    
    # Schritt 1: Initialisierung (t=0)
    print("Schritt 1: Initialisierung")
    for state in states:
        V[0][state] = start_prob[state] * emit_prob[state][observations[0]]
        path[0][state] = None
        print(f"V[0][{state}] = {start_prob[state]:.2f} * {emit_prob[state][observations[0]]:.2f} = {V[0][state]:.4f}")
    print()
    
    # Schritt 2: Rekursion (t=1 bis T-1)
    for t in range(1, T):
        print(f"Schritt 2: Rekursion für t={t}")
        
        for curr_state in states:
            # Finde den besten Vorgänger
            best_prob = 0
            best_prev = None
            
            for prev_state in states:
                prob = V[t-1][prev_state] * trans_prob[prev_state][curr_state]
                if prob > best_prob:
                    best_prob = prob
                    best_prev = prev_state
            
            # Multipliziere mit Emissionswahrscheinlichkeit
            V[t][curr_state] = best_prob * emit_prob[curr_state][observations[t]]
            path[t][curr_state] = best_prev
            
            print(f"V[{t}][{curr_state}] = {best_prob:.4f} * {emit_prob[curr_state][observations[t]]:.2f} = {V[t][curr_state]:.4f} (von {best_prev})")
        print()
    
    # Schritt 3: Terminierung
    print("Schritt 3: Terminierung")
    best_final_prob = 0
    best_final_state = None
    
    for state in states:
        if V[T-1][state] > best_final_prob:
            best_final_prob = V[T-1][state]
            best_final_state = state
    
    print(f"Beste Endwahrscheinlichkeit: {best_final_prob:.4f} in Zustand {best_final_state}")
    print()
    
    # Schritt 4: Pfad-Rekonstruktion
    print("Schritt 4: Pfad-Rekonstruktion")
    optimal_path = []
    current_state = best_final_state
    
    for t in range(T-1, -1, -1):
        optimal_path.append(current_state)
        current_state = path[t][current_state]
    
    optimal_path.reverse()
    
    print(f"Optimaler Pfad: {' → '.join(optimal_path)}")
    print(f"Wahrscheinlichkeit: {best_final_prob:.4f}")
    
    return optimal_path, best_final_prob

# Beispiel-Daten für Demonstration
print("📝 BEISPIEL-SETUP:")
print("HMM mit 2 Zuständen (Sonnig, Regnerisch) und 3 Beobachtungen (Spaziergang, Einkaufen, Putzen)")
print()

# Beispiel ausführen (nur wenn du möchtest)
example_obs = ['Spaziergang', 'Einkaufen', 'Putzen']
example_states = ['Sonnig', 'Regnerisch'] 
example_start = {'Sonnig': 0.6, 'Regnerisch': 0.4}
example_trans = {
    'Sonnig': {'Sonnig': 0.7, 'Regnerisch': 0.3},
    'Regnerisch': {'Sonnig': 0.4, 'Regnerisch': 0.6}
}
example_emit = {
    'Sonnig': {'Spaziergang': 0.6, 'Einkaufen': 0.3, 'Putzen': 0.1},
    'Regnerisch': {'Spaziergang': 0.1, 'Einkaufen': 0.4, 'Putzen': 0.5}
}

print("Ausführung mit: viterbi_demo(example_obs, example_start, example_trans, example_emit, example_states)")

📝 BEISPIEL-SETUP:
HMM mit 2 Zuständen (Sonnig, Regnerisch) und 3 Beobachtungen (Spaziergang, Einkaufen, Putzen)

Ausführung mit: viterbi_demo(example_obs, example_start, example_trans, example_emit, example_states)


### 🤔 Ist der Viterbi-Algorithmus "greedy"?

**NEIN! Der Viterbi-Algorithmus ist NICHT greedy!**

#### **Unterschied: Greedy vs. Viterbi**

**Greedy-Algorithmus würde:**
- In jedem Zeitschritt die **lokal beste** Entscheidung treffen
- Nur auf den aktuellen Zustand schauen
- **Kurzsichtig** handeln

**Viterbi-Algorithmus macht:**
- **Globale Optimierung** durch dynamische Programmierung
- Berücksichtigt **alle möglichen Pfade** bis zum aktuellen Zeitpunkt
- Findet den **global optimalen** Pfad

#### **Warum ist Viterbi besser?**

**Beispiel**: Gegeben die Beobachtungen [A, B, C]

**Greedy würde sagen:**
- t=1: "Welcher Zustand passt am besten zu A?" → Z1 (lokal optimal)
- t=2: "Welcher Zustand passt am besten zu B ausgehend von Z1?" → Z2
- t=3: "Welcher Zustand passt am besten zu C ausgehend von Z2?" → Z3
- **Ergebnis**: Z1 → Z2 → Z3 (möglicherweise suboptimal)

**Viterbi macht:**
- t=1: Berechne alle Möglichkeiten für alle Zustände
- t=2: Für jeden Zustand: Finde den besten Pfad der Länge 2
- t=3: Für jeden Zustand: Finde den besten Pfad der Länge 3
- **Ergebnis**: Global optimaler Pfad (z.B. Z2 → Z1 → Z3)

#### **Kernpunkt:**
Viterbi verwendet **dynamische Programmierung** und das **Optimalitätsprinzip**:
*"Der beste Pfad zu einem Punkt muss aus dem besten Pfad zum vorherigen Punkt bestehen"*

Das ist **nicht greedy**, sondern **optimal**!

In [19]:
# 🚀 KONKRETES BEISPIEL: Viterbi-Algorithmus in Aktion

print("🌟 BEISPIEL: Wettervorhersage mit Aktivitäten")
print("=" * 60)
print()

# Szenario: Basierend auf Aktivitäten (Spaziergang, Einkaufen, Putzen)
# wollen wir das Wetter (Sonnig, Regnerisch) vorhersagen

# Beispiel-Daten
observations = ['Spaziergang', 'Einkaufen', 'Putzen']
states = ['Sonnig', 'Regnerisch']

# Startwahrscheinlichkeiten
start_prob = {
    'Sonnig': 0.6,      # 60% Chance, dass es sonnig startet
    'Regnerisch': 0.4   # 40% Chance, dass es regnerisch startet
}

# Übergangswahrscheinlichkeiten
trans_prob = {
    'Sonnig': {
        'Sonnig': 0.7,      # 70% Chance: Sonnig → Sonnig
        'Regnerisch': 0.3   # 30% Chance: Sonnig → Regnerisch
    },
    'Regnerisch': {
        'Sonnig': 0.4,      # 40% Chance: Regnerisch → Sonnig
        'Regnerisch': 0.6   # 60% Chance: Regnerisch → Regnerisch
    }
}

# Emissionswahrscheinlichkeiten
emit_prob = {
    'Sonnig': {
        'Spaziergang': 0.6,  # Bei Sonnenwetter: 60% Spaziergang
        'Einkaufen': 0.3,    # Bei Sonnenwetter: 30% Einkaufen
        'Putzen': 0.1        # Bei Sonnenwetter: 10% Putzen
    },
    'Regnerisch': {
        'Spaziergang': 0.1,  # Bei Regen: 10% Spaziergang
        'Einkaufen': 0.4,    # Bei Regen: 40% Einkaufen
        'Putzen': 0.5        # Bei Regen: 50% Putzen
    }
}

print("📊 PARAMETER:")
print(f"Beobachtungen: {observations}")
print(f"Zustände: {states}")
print(f"Startwahrscheinlichkeiten: {start_prob}")
print(f"Übergangswahrscheinlichkeiten: {trans_prob}")
print(f"Emissionswahrscheinlichkeiten: {emit_prob}")
print()

# Führe Viterbi-Algorithmus aus
print("🔥 VITERBI-ALGORITHMUS AUSFÜHRUNG:")
print("=" * 60)
optimal_path, probability = viterbi_demo(observations, start_prob, trans_prob, emit_prob, states)

print()
print("🎯 ERGEBNIS:")
print(f"Wahrscheinlichste Wettersequenz: {' → '.join(optimal_path)}")
print(f"Gesamtwahrscheinlichkeit: {probability:.4f}")
print()
print("📝 INTERPRETATION:")
print("Tag 1: Spaziergang → Wahrscheinlich Sonnig")
print("Tag 2: Einkaufen → Wahrscheinlich Regnerisch")  
print("Tag 3: Putzen → Wahrscheinlich Regnerisch")
print()
print("💡 Das macht Sinn! Putzen und Einkaufen sind typische Regen-Aktivitäten!")

🌟 BEISPIEL: Wettervorhersage mit Aktivitäten

📊 PARAMETER:
Beobachtungen: ['Spaziergang', 'Einkaufen', 'Putzen']
Zustände: ['Sonnig', 'Regnerisch']
Startwahrscheinlichkeiten: {'Sonnig': 0.6, 'Regnerisch': 0.4}
Übergangswahrscheinlichkeiten: {'Sonnig': {'Sonnig': 0.7, 'Regnerisch': 0.3}, 'Regnerisch': {'Sonnig': 0.4, 'Regnerisch': 0.6}}
Emissionswahrscheinlichkeiten: {'Sonnig': {'Spaziergang': 0.6, 'Einkaufen': 0.3, 'Putzen': 0.1}, 'Regnerisch': {'Spaziergang': 0.1, 'Einkaufen': 0.4, 'Putzen': 0.5}}

🔥 VITERBI-ALGORITHMUS AUSFÜHRUNG:
🔍 VITERBI-ALGORITHMUS DEMONSTRATION
Beobachtungen: ['Spaziergang', 'Einkaufen', 'Putzen']
Zustände: ['Sonnig', 'Regnerisch']

Schritt 1: Initialisierung
V[0][Sonnig] = 0.60 * 0.60 = 0.3600
V[0][Regnerisch] = 0.40 * 0.10 = 0.0400

Schritt 2: Rekursion für t=1
V[1][Sonnig] = 0.2520 * 0.30 = 0.0756 (von Sonnig)
V[1][Regnerisch] = 0.1080 * 0.40 = 0.0432 (von Sonnig)

Schritt 2: Rekursion für t=2
V[2][Sonnig] = 0.0529 * 0.10 = 0.0053 (von Sonnig)
V[2][Regnerisch